# Unit 4 — Notebook 1: Multimodal Models
## CLIP, BLIP & Whisper

---

### What you will learn

| Model | Modalities | Key Capability |
|---|---|---|
| **CLIP** | Image + Text | Zero-shot image classification, image-text similarity |
| **BLIP** | Image + Text | Image captioning, Visual Question Answering (VQA) |
| **Whisper** | Audio + Text | Speech-to-text transcription, language detection |

### Why Multimodal?

The real world is not text-only. Humans process images, sound, and language simultaneously. Until recently, most ML models were **unimodal** — one model for images, another for text, another for audio. That created brittle pipelines requiring manual glue code between each step.

**Multimodal models** learn a **shared representation space** where different modalities (image, text, audio) can be directly compared, combined, and reasoned about — no glue code needed.

```
The Evolution:

  [Image Model]  +  [Text Model]  +  [Audio Model]
       ↓               ↓                ↓
  separate outputs, manual integration, brittle

  ↓↓↓  Multimodal Era  ↓↓↓

  [Image] ──┐
  [Text]  ──┼──► Shared Embedding Space ──► Unified Reasoning
  [Audio] ──┘
```

Real-world applications built on these three models:
- **CLIP**: Google Lens, Pinterest visual search, content moderation
- **BLIP**: Alt-text generation, accessibility tools, e-commerce product descriptions
- **Whisper**: Meeting transcription, podcast search, voice assistants


## Setup

All models in this notebook run locally via Hugging Face Transformers — **no API key needed**.

> **Colab users**: Runtime → Change runtime type → T4 GPU (recommended for Whisper)

In [ ]:
!pip install -q transformers torch torchvision Pillow soundfile datasets requests

In [ ]:
# Verify GPU availability (optional but speeds up inference)
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CPU mode — inference will be slower but fully functional.")

---

## Part 1: CLIP — Connecting Language and Images

### What is CLIP?

**CLIP** (Contrastive Language-Image Pretraining) was introduced by OpenAI in 2021. It was trained on **400 million image-text pairs** scraped from the internet using a **contrastive objective**: push matching image-text pairs closer together in embedding space, and push non-matching pairs apart.

```
Training objective (Contrastive Learning):

  "a photo of a cat" ──► Text Encoder ──► text_embedding
       [cat image]  ──► Image Encoder ──► image_embedding

  Loss = maximize cosine_similarity(text_embedding, image_embedding)
         for MATCHING pairs
         minimize it for NON-MATCHING pairs
```

### The Zero-Shot Superpower

Because CLIP learned from natural language descriptions, it can classify images into categories it has **never explicitly been trained on** — just by comparing an image's embedding against text prompt embeddings like `"a photo of a {class}"`.

No fine-tuning. No labeled training data. Just prompts.

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests
import torch
import io

# ── Robust image loader ───────────────────────────────────────────────────────
# Many hosts (Wikipedia, Wikimedia) block raw streaming requests without a
# browser-like User-Agent. This helper handles that reliably.
def load_image(url: str) -> Image.Image:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; NotebookDemo/1.0)"}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return Image.open(io.BytesIO(response.content)).convert("RGB")

print("Loading CLIP model (openai/clip-vit-base-patch32)...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP loaded.")

### 1.1 Image-Text Similarity Scoring

Let's start with the core operation: given an image and a list of text descriptions, compute how well each text matches the image.

In [ ]:
# Load a sample image (we'll use a publicly available image)
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"  # two cats on a couch
image = load_image(image_url)

# Display the image
image

In [ ]:
# Define candidate text descriptions
candidate_texts = [
    "a photo of cats sleeping on a couch",
    "a photo of a dog playing in the park",
    "a photo of a car on a highway",
    "a photo of two cats resting indoors",
    "a photo of a bird flying in the sky"
]

# Preprocess inputs
inputs = clip_processor(
    text=candidate_texts,
    images=image,
    return_tensors="pt",
    padding=True
).to(device)

# Forward pass
with torch.no_grad():
    outputs = clip_model(**inputs)

# logits_per_image: similarity scores [1 image x N texts]
logits = outputs.logits_per_image  # shape: (1, 5)
probs = logits.softmax(dim=1).squeeze().tolist()

print("CLIP Image-Text Similarity Scores:")
print("-" * 55)
for text, prob in sorted(zip(candidate_texts, probs), key=lambda x: -x[1]):
    bar = "█" * int(prob * 40)
    print(f"{prob:.1%}  {bar}")
    print(f"       '{text}'")

### 1.2 Zero-Shot Image Classification

Now let's use CLIP as a **zero-shot classifier** — a task no traditional CNN was designed for.

We use the standard CLIP prompt template: `"a photo of a {label}"` for each class.

In [ ]:
def clip_zero_shot_classify(image, labels, model, processor, device="cpu"):
    """
    Zero-shot classification using CLIP.

    Args:
        image: PIL Image
        labels: list of class names (e.g. ["cat", "dog", "car"])
        model, processor: loaded CLIP model/processor
        device: "cpu" or "cuda"

    Returns:
        dict of {label: probability}
    """
    # Convert labels to CLIP-style prompts
    prompts = [f"a photo of a {label}" for label in labels]

    inputs = processor(
        text=prompts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=1).squeeze().tolist()
    return dict(zip(labels, probs))


# Test on our cat image
labels = ["cat", "dog", "bird", "car", "airplane", "sofa", "person"]
results = clip_zero_shot_classify(image, labels, clip_model, clip_processor, device)

print("Zero-Shot Classification Results:")
print("-" * 40)
for label, prob in sorted(results.items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 40)
    print(f"{label:<10} {prob:.1%}  {bar}")

In [ ]:
# Try with a different image — let's test a dog image
dog_image_url = "https://upload.wikimedia.org/wikipedia/commons/2/26/YellowLabradorLooking_new.jpg"
dog_image = load_image(dog_image_url)

results_dog = clip_zero_shot_classify(dog_image, labels, clip_model, clip_processor, device)

print("Zero-Shot Classification — Dog image:")
print("-" * 40)
for label, prob in sorted(results_dog.items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 40)
    print(f"{label:<10} {prob:.1%}  {bar}")

### 1.3 Image Embeddings — The Shared Vector Space

The real power of CLIP is the **shared embedding space**. Both images and text live in the same 512-dimensional space. Let's extract and compare raw embeddings.

In [ ]:
import torch.nn.functional as F

def get_clip_image_embedding(image, model, processor, device):
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        embedding = model.get_image_features(**inputs)
        if not isinstance(embedding, torch.Tensor):
            embedding = embedding.pooler_output
    return F.normalize(embedding, dim=-1)

def get_clip_text_embedding(text, model, processor, device):
    inputs = processor(text=[text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        embedding = model.get_text_features(**inputs)
        if not isinstance(embedding, torch.Tensor):
            embedding = embedding.pooler_output
    return F.normalize(embedding, dim=-1)

# Compare: cat image vs. text descriptions
cat_emb = get_clip_image_embedding(image, clip_model, clip_processor, device)

comparisons = [
    "two cats sleeping on a sofa",
    "a dog running in a park",
    "feline animals resting indoors",  # paraphrase — tests semantic understanding
    "a vehicle on a road"
]

print("Cosine similarity: cat image vs. text descriptions")
print("-" * 55)
for text in comparisons:
    text_emb = get_clip_text_embedding(text, clip_model, clip_processor, device)
    sim = (cat_emb @ text_emb.T).item()
    bar = "█" * int((sim + 1) * 20)  # normalize to positive range
    print(f"{sim:.4f}  {bar}")
    print(f"        '{text}'")

**Key insight**: Notice that `"feline animals resting indoors"` scores high despite not containing the word "cat". CLIP understands **semantics**, not just keywords — because it learned from natural language descriptions.

---

## Part 2: BLIP — Bootstrapped Language-Image Pretraining

### What is BLIP?

**BLIP** (Bootstrapping Language-Image Pretraining), introduced by Salesforce in 2022, goes beyond CLIP's similarity matching to actually **generate text from images**.

While CLIP asks: *"How similar is this image to this text?"*
BLIP asks: *"What text best describes this image?"* and *"Given this image and question, what is the answer?"*

```
CLIP:  Image + Text ──► Similarity Score  (discriminative)
BLIP:  Image ──► Generated Caption        (generative)
       Image + Question ──► Answer        (generative VQA)
```

BLIP uses a clever **bootstrapping** technique: it uses its own captioning model to generate synthetic captions for noisy web images, then filters them using an Image-Text Matching (ITM) module, creating a cleaner training set automatically.

### Two Key Tasks
1. **Image Captioning** — Generate a natural language description of an image
2. **Visual Question Answering (VQA)** — Answer questions about an image

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration, BlipForQuestionAnswering

print("Loading BLIP Captioning model...")
blip_caption_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_caption_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)
print("BLIP Captioning loaded.")

### 2.1 Image Captioning

In [ ]:
def blip_caption(image, model, processor, device, conditional_text=None):
    """
    Generate a caption for an image.

    Args:
        image: PIL Image
        conditional_text: optional text prefix to condition the caption
                          e.g. "a photography of" — guides the style
    """
    if conditional_text:
        inputs = processor(image, conditional_text, return_tensors="pt").to(device)
    else:
        inputs = processor(image, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50)

    return processor.decode(output[0], skip_special_tokens=True)


# Caption the cat image (unconditional)
caption_unconditional = blip_caption(image, blip_caption_model, blip_caption_processor, device)
print(f"Unconditional caption:  {caption_unconditional}")

# Caption with a conditional prefix
caption_conditional = blip_caption(
    image, blip_caption_model, blip_caption_processor, device,
    conditional_text="a photography of"
)
print(f"Conditional caption:    {caption_conditional}")

In [ ]:
# Test on multiple images to see the range of BLIP's captioning
test_images = {
    "cats on couch": image,
    "labrador dog": dog_image,
}

# Add a third image — a landscape
landscape_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/24701-nature-natural-beauty.jpg/1280px-24701-nature-natural-beauty.jpg"
try:
    landscape_image = load_image(landscape_url)
    test_images["landscape"] = landscape_image
except:
    print("Could not load landscape image, continuing with 2 images.")

print("BLIP Captions:")
print("-" * 50)
for name, img in test_images.items():
    caption = blip_caption(img, blip_caption_model, blip_caption_processor, device)
    print(f"[{name}]")
    print(f"  Caption: {caption}")
    print()

### 2.2 Visual Question Answering (VQA)

VQA is more challenging: the model must understand the image **and** interpret a natural language question to produce a correct answer.

In [ ]:
print("Loading BLIP VQA model...")
blip_vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
blip_vqa_model = BlipForQuestionAnswering.from_pretrained(
    "Salesforce/blip-vqa-base"
).to(device)
print("BLIP VQA loaded.")

In [ ]:
def blip_vqa(image, question, model, processor, device):
    """
    Answer a question about an image.

    Args:
        image: PIL Image
        question: str — natural language question about the image

    Returns:
        str — model's answer
    """
    inputs = processor(image, question, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=20)
    return processor.decode(output[0], skip_special_tokens=True)


# Ask multiple questions about the cat image
questions = [
    "How many cats are in the image?",
    "What are the cats doing?",
    "What color are the cats?",
    "Where are the cats sitting?",
    "Is it daytime or nighttime?"
]

print("VQA — Cats on couch image:")
print("-" * 50)
for question in questions:
    answer = blip_vqa(image, question, blip_vqa_model, blip_vqa_processor, device)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

In [ ]:
# VQA on the dog image
dog_questions = [
    "What breed of dog is this?",
    "What color is the dog?",
    "Is the dog indoors or outdoors?"
]

print("VQA — Labrador image:")
print("-" * 50)
for question in dog_questions:
    answer = blip_vqa(dog_image, question, blip_vqa_model, blip_vqa_processor, device)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

---

## Part 3: Whisper — Universal Speech Recognition

### What is Whisper?

**Whisper** is an automatic speech recognition (ASR) system developed by OpenAI, released in 2022. It was trained on **680,000 hours of multilingual audio** from the internet, making it one of the most robust ASR models available.

Key properties:
- **Multilingual**: 99 languages supported
- **Zero-shot**: no fine-tuning needed for most languages
- **Robust**: handles accents, background noise, technical jargon
- **Tasks**: transcription, translation (to English), language identification

```
Whisper Architecture:

  Audio Waveform
       ↓
  Log-Mel Spectrogram  (80 frequency bins, 30s chunks)
       ↓
  Convolutional Feature Extractor
       ↓
  Transformer Encoder  (processes audio features)
       ↓
  Transformer Decoder  (generates text tokens autoregressively)
       ↓
  Transcript / Translation
```

### Model Sizes

| Model | Parameters | VRAM | Speed |
|---|---|---|---|
| tiny | 39M | ~1 GB | fastest |
| base | 74M | ~1 GB | fast |
| small | 244M | ~2 GB | balanced |
| medium | 769M | ~5 GB | accurate |
| large | 1550M | ~10 GB | most accurate |

We'll use `whisper-base` — fast enough for demos, accurate enough to be impressive.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import soundfile as sf
import numpy as np

print("Loading Whisper model (openai/whisper-base)...")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-base")
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-base"
).to(device)
print("Whisper loaded.")

### 3.1 Transcription from a Sample Audio File

We'll download a sample English speech audio file from a public dataset.

In [ ]:
from datasets import load_dataset, Audio
import soundfile as sf
import io

# Load the dataset with decode=False so HuggingFace never calls torchcodec.
# We decode audio ourselves with soundfile — works on all platforms.
print("Loading a sample audio file from LibriSpeech...")
ds = load_dataset(
    "hf-internal-testing/librispeech_asr_demo",
    "clean",
    split="validation"
).cast_column("audio", Audio(decode=False))  # raw bytes, no torchcodec needed

def load_audio_sample(sample):
    """Decode a HuggingFace audio sample (decode=False) using soundfile."""
    audio_bytes = sample["audio"]["bytes"]
    audio_array, sampling_rate = sf.read(io.BytesIO(audio_bytes))
    return audio_array.astype("float32"), sampling_rate

sample = ds[0]
audio_array, sampling_rate = load_audio_sample(sample)

print(f"Audio sample loaded.")
print(f"  Duration:      {len(audio_array) / sampling_rate:.2f} seconds")
print(f"  Sampling rate: {sampling_rate} Hz")
print(f"  Reference transcript (ground truth): '{sample['text']}'")

In [ ]:
def whisper_transcribe(audio_array, sampling_rate, model, processor, device, task="transcribe"):
    """
    Transcribe (or translate) audio using Whisper.

    Args:
        audio_array: numpy array of audio samples (float32)
        sampling_rate: int — Hz of the audio (Whisper expects 16000 Hz)
        task: "transcribe" or "translate" (translate converts to English)

    Returns:
        str — transcribed / translated text
    """
    # Whisper requires 16kHz audio
    if sampling_rate != 16000:
        import librosa
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=16000)
        sampling_rate = 16000

    # Extract log-mel spectrogram features
    input_features = processor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to(device)

    # Generate transcription
    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="english", task=task
    )

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids
        )

    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]


# Transcribe
transcript = whisper_transcribe(audio_array, sampling_rate, whisper_model, whisper_processor, device)

print("Whisper Transcription Result:")
print("-" * 50)
print(f"Whisper output:  '{transcript}'")
print(f"Ground truth:    '{sample['text']}'")

### 3.2 Language Detection

Whisper can detect what language is being spoken — before even starting to transcribe.

In [ ]:
def whisper_detect_language(audio_array, sampling_rate, model, processor, device):
    """
    Detect the spoken language using Whisper.
    Returns the detected language code and confidence scores for top-5 languages.
    """
    if sampling_rate != 16000:
        import librosa
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=16000)

    input_features = processor(
        audio_array,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        encoder_output = model.model.encoder(input_features)
        decoder_input = torch.tensor([[model.config.decoder_start_token_id]]).to(device)
        logits = model(
            input_features=input_features,
            decoder_input_ids=decoder_input
        ).logits

    # Get language tokens from the tokenizer's vocab directly
    tokenizer = processor.tokenizer
    lang_tokens = {
        token: tokenizer.convert_tokens_to_ids(token)
        for token in tokenizer.all_special_tokens
        if token.startswith("<|") and token.endswith("|>")
        and token not in ("<|endoftext|>", "<|startoftranscript|>",
                          "<|translate|>", "<|transcribe|>",
                          "<|startoflm|>", "<|startofprev|>",
                          "<|nospeech|>", "<|notimestamps|>")
    }

    token_list = list(lang_tokens.keys())
    token_ids = list(lang_tokens.values())

    lang_logits = logits[0, 0, token_ids]
    lang_probs = torch.softmax(lang_logits, dim=-1)

    top5_probs, top5_indices = torch.topk(lang_probs, 5)
    top5_langs = [token_list[i] for i in top5_indices.tolist()]
    top5_scores = top5_probs.tolist()

    return list(zip(top5_langs, top5_scores))


lang_results = whisper_detect_language(audio_array, sampling_rate, whisper_model, whisper_processor, device)

print("Language Detection — Top 5:")
print("-" * 40)
for lang_token, prob in lang_results:
    lang_name = lang_token.strip("<|>").replace("|", "")
    bar = "█" * int(prob * 50)
    print(f"{lang_name:<12} {prob:.2%}  {bar}")

### 3.3 Transcribing Multiple Speakers / Samples

Let's batch-process multiple audio samples to see Whisper's consistency.

In [ ]:
# Transcribe the first 3 samples from the dataset
print("Batch transcription — 3 samples:")
print("=" * 60)

for i, sample in enumerate(ds.select(range(3))):
    audio, sr = load_audio_sample(sample)

    transcript = whisper_transcribe(audio, sr, whisper_model, whisper_processor, device)
    reference = sample["text"]

    print(f"Sample {i+1}:")
    print(f"  Whisper:    {transcript}")
    print(f"  Reference:  {reference}")
    # Simple word accuracy
    w_words = set(transcript.lower().split())
    r_words = set(reference.lower().split())
    overlap = len(w_words & r_words) / max(len(r_words), 1)
    print(f"  Word overlap: {overlap:.0%}")
    print()

---

## Part 4: Putting It All Together — A Multimodal Pipeline

### The Pipeline

Now let's combine all three models into a single pipeline that demonstrates cross-modal reasoning:

```
MULTIMODAL PIPELINE
───────────────────────────────────────────────────────

  [Audio: "Show me pictures of cats"]                   
         ↓  Whisper                                     
  [Text Query: "Show me pictures of cats"]              
         ↓  CLIP (text encoder)                         
  [Text Embedding]                                      
         ↓  Compare against image embeddings            
  [Ranked Images]  ← CLIP (image encoder) ← [Image DB]
         ↓  BLIP                                        
  [Caption of top result]                               
───────────────────────────────────────────────────────
```

This is the core of many real-world systems:
- Voice-based image search (Google Lens with voice)
- Accessibility tools (describe what you're looking at by speaking)
- Content moderation pipelines

In [ ]:
# Build a mini image database (5 images with different subjects)
image_db = {}

image_urls = {
    "cats_on_couch": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "labrador_dog":  "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/640px-YellowLabradorLooking_new.jpg",
    "red_car":       "http://images.cocodataset.org/val2017/000000252219.jpg",
    "pizza":         "http://images.cocodataset.org/val2017/000000397133.jpg",
    "people_street": "http://images.cocodataset.org/val2017/000000037777.jpg",
}

print("Building image database...")
for name, url in image_urls.items():
    try:
        img = load_image(url)
        image_db[name] = img
        print(f"  Loaded: {name}")
    except Exception as e:
        print(f"  Failed to load {name}: {e}")

print(f"\nImage database ready: {len(image_db)} images")

In [ ]:
# Pre-compute CLIP embeddings for all images in the database
print("Pre-computing CLIP image embeddings...")
db_embeddings = {}
for name, img in image_db.items():
    db_embeddings[name] = get_clip_image_embedding(img, clip_model, clip_processor, device)
print("Embeddings computed.")


def multimodal_search(text_query, db_embeddings, image_db, clip_model, clip_processor,
                      blip_caption_model, blip_caption_processor, device, top_k=2):
    """
    Full multimodal pipeline:
    1. Encode the text query with CLIP
    2. Find the most similar images by cosine similarity
    3. Generate BLIP captions for the top results
    """
    # Step 1: CLIP text embedding
    query_emb = get_clip_text_embedding(text_query, clip_model, clip_processor, device)

    # Step 2: Rank images by similarity
    scores = {}
    for name, img_emb in db_embeddings.items():
        scores[name] = (query_emb @ img_emb.T).item()

    ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_k]

    # Step 3: BLIP captions for top results
    results = []
    for name, score in ranked:
        caption = blip_caption(
            image_db[name], blip_caption_model, blip_caption_processor, device
        )
        results.append({"name": name, "similarity": score, "caption": caption})

    return results


# Run text-based queries
queries = [
    "a domestic animal",
    "food on a plate",
    "people walking outdoors"
]

print("Multimodal Image Search Results:")
print("=" * 60)
for query in queries:
    results = multimodal_search(
        query, db_embeddings, image_db,
        clip_model, clip_processor,
        blip_caption_model, blip_caption_processor,
        device, top_k=2
    )
    print(f"\nQuery: '{query}'")
    for r in results:
        print(f"  [{r['name']}]  similarity={r['similarity']:.4f}")
        print(f"  BLIP caption: {r['caption']}")

In [ ]:
# Now simulate the FULL pipeline: audio → text → image search → caption
# We use the LibriSpeech audio and connect it to our image search

# Step 1: Whisper transcribes the audio
spoken_query_audio, spoken_query_sr = load_audio_sample(ds[0])

print("Step 1 — Whisper: Transcribing audio...")
spoken_text = whisper_transcribe(spoken_query_audio, spoken_query_sr,
                                  whisper_model, whisper_processor, device)
print(f"  Transcribed: '{spoken_text}'")

# Step 2: CLIP finds the most relevant image
print("\nStep 2 — CLIP: Finding most relevant image...")
results = multimodal_search(
    spoken_text, db_embeddings, image_db,
    clip_model, clip_processor,
    blip_caption_model, blip_caption_processor,
    device, top_k=1
)

# Step 3: BLIP captions the result
print(f"\nStep 3 — BLIP: Caption of top result [{results[0]['name']}]:")
print(f"  {results[0]['caption']}")
print(f"  (similarity score: {results[0]['similarity']:.4f})")

print("\n" + "=" * 60)
print("Pipeline complete:")
print(f"  Audio → '{spoken_text}'")
print(f"  Best image match: {results[0]['name']}")
print(f"  Caption: {results[0]['caption']}")

---

## Summary

| Model | Core Idea | What we built |
|---|---|---|
| **CLIP** | Shared image-text embedding space via contrastive learning | Zero-shot classifier, similarity scorer, image search engine |
| **BLIP** | Generative vision-language model with bootstrapped training | Image captioning, Visual Q&A |
| **Whisper** | Transformer encoder-decoder trained on 680k hours of audio | Transcription, language detection |
| **Pipeline** | Combining all three | Audio query → text → image retrieval → caption |

### Key Takeaways

1. **CLIP** is discriminative — it scores similarity. It cannot generate text from scratch.
2. **BLIP** is generative — it can produce captions and answers, but it builds on CLIP-style alignment.
3. **Whisper** is specialized for audio — it converts the audio modality into text, making it the entry point for voice-driven pipelines.
4. **Multimodal pipelines** chain these models: each model handles its modality and passes outputs to the next.

### What's Next

In **Notebook 2** we'll look at Agentic Workflows — where LLMs don't just respond but plan, use tools, and collaborate with other agents to complete complex tasks.